# polars and data types

It is worth looking into how polars stores data internally by default, and how that behaviour can be modified if necessary.

In general, polars is rather liberal with respect to the data types it uses, i.e., integers are by default represented by 64 bit, and floating point values as double precision numbers.  Since one of the performance bottlenecks is often memory bandwidth, and memory may be at a premium when dealing with large datasets, it is often required to ensure that the columns of dataframes have more appropriate types for the data they have to store.

# Data generation

In order to do some experiments, we generate a data set.  We want to experiment with various types of floating point numbers, integers and categorical data, so we will create a text file that has some columns of each of those types.

In [1]:
import pathlib
import random

In [2]:
file_path = pathlib.Path('random_data_to_remove.csv')
nr_lines = 100_000

In [3]:
with open(file_path, 'w') as file:
    categories = ''.join(chr(ord('A') + i) for i in range(26))
    print('float1,float2,int1,int2,binary1,binary2,cat1,cat2',
          file=file)
    for _ in range(nr_lines):
        print(f'{random.random()},{random.random()},'
              f'{random.randrange(0, 255)},{random.randrange(0, 255)},'
              f'{random.randint(0, 1)},{random.randint(0, 1)},'
              f'{random.choice(categories)},{random.choice(categories)}',
              file=file)

# Tailoring dataframes

In [4]:
import sys
import polars as pl

When simply using defaults to read the CSV file, polars will use 64-bit floating point and integer values, even though that might not be required.

In [5]:
data = pl.read_csv(file_path)

In [6]:
data.schema

Schema([('float1', Float64),
        ('float2', Float64),
        ('int1', Int64),
        ('int2', Int64),
        ('binary1', Int64),
        ('binary2', Int64),
        ('cat1', String),
        ('cat2', String)])

The size of a dataframe can be queried.

In [7]:
data.estimated_size('mb')

4.76837158203125

Of course one can convert columns of an existing dataframe info more frugal data types, e.g.,

In [8]:
data = data.with_columns(pl.col('float1').cast(pl.Float32))

In [9]:
data.schema

Schema([('float1', Float32),
        ('float2', Float64),
        ('int1', Int64),
        ('int2', Int64),
        ('binary1', Int64),
        ('binary2', Int64),
        ('cat1', String),
        ('cat2', String)])

In [10]:
data.estimated_size('mb')

4.38690185546875

However, it is faster to specify a dictionary of data types when reading the file. Not only will this reduce the execution time, but it will also allow to deal with larger data sets since they do not require the initial (overlarge) memory footprint.

Specifying the type of the floating point values as single precesion reduces the size of the dataframe considerably.

In [11]:
data = pl.read_csv(
    file_path,
    schema_overrides={
       'float1': pl.Float32,
       'float2': pl.Float32,
    },
)

In [12]:
data.schema

Schema([('float1', Float32),
        ('float2', Float32),
        ('int1', Int64),
        ('int2', Int64),
        ('binary1', Int64),
        ('binary2', Int64),
        ('cat1', String),
        ('cat2', String)])

In [13]:
data.estimated_size('mb')

4.00543212890625

Similarly, the type of the integer data can be reduced to `np.uint8`, further reducing the size of the dataframe.

In [14]:
data = pl.read_csv(
    file_path,
    schema_overrides={
        'float1': pl.Float32, 'float2': pl.Float32,
        'int1': pl.UInt8, 'int2': pl.UInt8,
    },
)

In [15]:
data.schema

Schema([('float1', Float32),
        ('float2', Float32),
        ('int1', UInt8),
        ('int2', UInt8),
        ('binary1', Int64),
        ('binary2', Int64),
        ('cat1', String),
        ('cat2', String)])

In [16]:
data.estimated_size('mb')

2.6702880859375

Futhermore, both the columns with binary data and the categorical data can be explicitly represented as such, further reducing the memory footprint of the dataframe.

In [17]:
data = pl.read_csv(
    file_path,
    schema_overrides={
       'float1': pl.Float32, 'float2': pl.Float32,
       'int1': pl.UInt8, 'int2': pl.UInt8,
       'binary1': pl.Categorical, 'binary2': pl.Categorical,
       'cat1': pl.Categorical, 'cat2': pl.Categorical,
   },
)

In [18]:
data.schema

Schema([('float1', Float32),
        ('float2', Float32),
        ('int1', UInt8),
        ('int2', UInt8),
        ('binary1', Categorical),
        ('binary2', Categorical),
        ('cat1', Categorical),
        ('cat2', Categorical)])

In [19]:
data.estimated_size('mb')

2.47955322265625

Just to see how efficiently categories are represented, we load only a single binary column, and check the size of the dataframe.

In [20]:
data = pl.read_csv(
    file_path,
   columns=('binary1',),
   schema_overrides={
       'binary1': pl.Categorical,
    },
)

In [21]:
data.estimated_size('kb')

390.625

This doesn't seem very efficient as the data could be represented by a single byte.  Using a `pl.Enum` type with known values `'0'` and `'1'` is more efficient.

In [22]:
data = pl.read_csv(
    file_path,
   columns=('binary1',),
   schema_overrides={
       'binary1': pl.Enum(['0', '1']),
    },
)

In [23]:
data.estimated_size('kb')

97.65625

Loading just an integer column as `np.uint8` shows that again, the memory footprint is the same, so this is the data type used to represent categorical data with less than 256 categories.

In [24]:
data = pl.read_csv(
    file_path,
    columns=('int1',),
    schema_overrides={
        'int1': pl.UInt8,
    },
)

In [25]:
data.estimated_size('kb')

97.65625

In [26]:
file_path.unlink()

# Failures

You should note that when you specify the data types to read data and this doesn't correspond to the data, failures will be silent.  For instance, integers will overflow.

Consider the following dataframe that has two integer and a floating point column.  Column `B` contains integers that can not be represented as 8-bit integers, and column `C` contains values that can not be represented as single precision numbers.

In [27]:
df = pl.DataFrame({
    'A': [3, 19, 5, 7],
    'B': [9, 7, 283, -1],
    'C': [4.59, 21.49e57, -4.8, -1.495e-103]
})

We write this dataframe as a CSV file.

In [28]:
file_path = pathlib.Path('overflow_to_remove.csv')

In [29]:
df.write_csv(file_path)

When we read it back in, we specify the bypes of columns `A` and `B` as `np.uint8` and `C` as `np.float32`.

In [30]:
try:
    df_typed = pl.read_csv(
        file_path,
        schema={
            'A': pl.UInt8,
            'B': pl.UInt8,
            'C': pl.Float32
        },
    )
except Exception as e:
    print(e, file=sys.stderr)

could not parse `283` as dtype `u8` at column 'B' (column number 2)

The current offset in the file is 26 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- increasing `infer_schema_files` (e.g. `infer_schema_files=50`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `283` to the `null_values` list.

Original error: ```invalid primitive value found during CSV parsing```


The error is quite to the point: the integer 283 can not be converted into an unsigned integer using 8 bits.

Using explicit schemas in polars has the advantage of catching such mistakes that could lead to subtle bugs otherwise.

In [31]:
file_path.unlink()